In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 82.3 MB/s eta 0:00:00


In [2]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.6/329.6 kB 11.8 MB/s eta 0:00:00


In [3]:
!pip install langchain_openai
!pip install langchain_community
!pip install requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.7/84.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 19.2 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.1
    Uninstalling langchain-core-1.2.1:
      Successfully uninstalled langchain-core-1.2.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-co

In [4]:
!pip install langgraph

In [5]:
!pip install langchain.text_splitter

ERROR: Could not find a version that satisfies the requirement langchain.text_splitter (from versions: none)
ERROR: No matching distribution found for langchain.text_splitter


In [6]:
import os
import re
from typing import TypedDict, List
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import StateGraph, END
import requests

In [7]:
oapi_key = "REDACTED_CREDENTIAL"
ppl_key = "REDACTED_CREDENTIAL"

os.environ["OPENAI_API_KEY"] = oapi_key

In [8]:
# ============================================
# 상태 정의
# ============================================
class InvestmentState(TypedDict):
    user_query: str          # 사용자 질문 (원본 그대로)
    market_data: dict        # Perplexity 수집 데이터
    buffett_insights: List[str]  # RAG 검색 결과
    final_analysis: str      # 최종 분석 결과
    error: str

In [9]:
def perplexity_research_node(state: InvestmentState) -> InvestmentState:
    """Perplexity API로 신뢰할 수 있는 금융 사이트에서만 정보 수집"""
    user_query = state["user_query"]

    print(f"🔍 Perplexity로 정보 수집 중: '{user_query}'")

    try:
        # Perplexity API 호출
        url = "https://api.perplexity.ai/chat/completions"

        payload = {
            "model": "sonar-pro",
            "messages": [
                {
                    "role": "system",
                    "content": """You are a financial data researcher. When asked about a stock:
1. Identify the company and ticker symbol
2. Provide current stock price and today's change
3. Recent news (last 7 days)
4. Analyst ratings summary
5. Key financial metrics (P/E, market cap, revenue growth)
6. Major risks or concerns

Be concise and factual. Always include the ticker symbol in your response."""
                },
                {
                    "role": "user",
                    "content": user_query
                }
            ],
            "max_tokens": 1500,
            "temperature": 0.2,
            "search_domain_filter": [
                "bloomberg.com",
                "reuters.com",
                "wsj.com",
                "investing.com",
                "finance.yahoo.com"
            ]
        }

        headers = {
            "Authorization": f"Bearer {ppl_key}",
            "Content-Type": "application/json"
        }

        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()

        result = response.json()
        market_data = {
            "raw_response": result["choices"][0]["message"]["content"],
            "citations": result.get("citations", []),
            "user_query": user_query
        }

        print(f"✓ Perplexity 정보 수집 완료 (출처: 주요 금융 사이트)")

        return {
            **state,
            "market_data": market_data
        }

    except Exception as e:
        print(f"❌ Perplexity API 오류: {str(e)}")
        return {
            **state,
            "market_data": {
                "raw_response": f"정보 수집 실패: {str(e)}",
                "user_query": user_query
            },
            "error": str(e)
        }

In [10]:
# ============================================
# 노드 2: RAG - 버크셔 서한 검색
# ============================================
# 전역 벡터 스토어 (실전에서는 초기화 로직 분리)
vector_store = None

def initialize_rag(pdf_path: str):
    """RAG 시스템 초기화 (PDF 로드 및 벡터화)"""
    global vector_store

    print(f"📄 PDF 로딩 중: {pdf_path}")

    # PDF 로드
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # 텍스트 분할
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    splits = text_splitter.split_documents(documents)

    # 벡터 스토어 생성
    embeddings = OpenAIEmbeddings()
    vector_store = FAISS.from_documents(splits, embeddings)

    print(f"✓ RAG 초기화 완료: {len(splits)}개 청크")

In [11]:
def rag_buffett_wisdom_node(state: InvestmentState) -> InvestmentState:
    """Perplexity 결과 기반으로 관련 버핏 철학 검색"""

    market_data = state["market_data"]
    perplexity_info = market_data.get('raw_response', '')

    print(f"📚 Perplexity 결과 분석 후 관련 철학 검색 중...")

    # OpenAI로 Perplexity 결과 분석 → 검색어 생성
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

    analysis_prompt = f"""다음은 특정 주식에 대한 정보입니다:

{perplexity_info}

이 회사를 평가하기 위해 워렌 버핏의 투자 서한에서 찾아야 할 주제 5개를 영어로 나열하세요.
예: "technology company investment", "high valuation stocks", "capital allocation"

주제만 쉼표로 구분해서 출력하세요."""

    response = llm.invoke(analysis_prompt)
    search_topics = [topic.strip() for topic in response.content.split(',')]

    print(f"  생성된 검색 주제: {search_topics}")

    # 동적 검색어 + 기본 원칙
    search_queries = search_topics + [
        "competitive advantage moat",
        "margin of safety"
    ]

    insights = []
    for query in search_queries[:7]:  # 최대 7개
        docs = vector_store.similarity_search(query, k=2)
        for doc in docs:
            insights.append(doc.page_content[:300])

    return {
        **state,
        "buffett_insights": insights
    }

In [12]:
# ============================================
# 노드 3: OpenAI 종합 분석
# ============================================
def openai_analysis_node(state: InvestmentState) -> InvestmentState:
    """OpenAI로 수집된 정보와 버핏 철학 기반 종합 분석"""
    user_query = state["user_query"]
    market_data = state["market_data"]
    buffett_insights = state["buffett_insights"]

    print(f"🤖 OpenAI로 종합 분석 중...")

    # 프롬프트 구성
    prompt = f"""당신은 워렌 버핏의 투자 철학을 깊이 이해하는 전문 애널리스트입니다.

## 사용자 질문
{user_query}

## Perplexity가 수집한 최신 시장 정보
{market_data.get('raw_response', '정보 없음')}

## 버크셔 해서웨이 서한에서 추출한 투자 원칙
{chr(10).join(f"- {insight[:200]}..." for insight in buffett_insights)}

## 분석 요청
위 정보를 바탕으로 다음 구조로 분석하세요:

1. **회사 및 티커 확인**
   - Perplexity 정보에서 정확한 회사명과 티커 명시

2. **비즈니스 이해도** (1-5점)
   - 이 회사의 비즈니스 모델이 명확한가?
   - 버핏의 "능력 범위" 원칙에 부합하는가?

3. **경제적 해자** (1-5점)
   - 지속 가능한 경쟁 우위가 있는가?
   - 브랜드, 네트워크 효과, 규모의 경제 등

4. **경영진 평가** (1-5점)
   - 주주 친화적 정책인가?
   - 자본 배분 능력

5. **밸류에이션** (1-5점)
   - 내재 가치 대비 현재 가격은?
   - 안전마진이 있는가?

6. **종합 투자 의견**
   - 버핏이라면 어떻게 판단할까?
   - 구체적인 액션 아이템 (매수/보유/매도/관망)
   - 주의사항 및 리스크

한국어로 명확하고 실용적으로 작성해주세요.
"""

    try:
        llm = ChatOpenAI(model="gpt-4o", temperature=0.3)
        response = llm.invoke(prompt)

        analysis = response.content

        print(f"✓ 분석 완료 ({len(analysis)} 글자)")

        return {
            **state,
            "final_analysis": analysis
        }

    except Exception as e:
        print(f"❌ OpenAI API 오류: {str(e)}")
        return {
            **state,
            "final_analysis": f"분석 중 오류 발생: {str(e)}",
            "error": str(e)
        }

In [13]:
# ============================================
# LangGraph 워크플로우 구성 (단순화)
# ============================================
def create_investment_workflow():
    """투자 분석 워크플로우 생성 (티커 추출 단계 제거)"""

    workflow = StateGraph(InvestmentState)

    # 노드 추가 (티커 추출 제거!)
    workflow.add_node("perplexity_research", perplexity_research_node)
    workflow.add_node("rag_wisdom", rag_buffett_wisdom_node)
    workflow.add_node("openai_analysis", openai_analysis_node)

    # 엣지 정의 (순차 실행)
    workflow.set_entry_point("perplexity_research")
    workflow.add_edge("perplexity_research", "rag_wisdom")
    workflow.add_edge("rag_wisdom", "openai_analysis")
    workflow.add_edge("openai_analysis", END)

    return workflow.compile()

In [14]:
# ============================================
# 메인 실행 함수
# ============================================
def analyze_stock(user_query: str, pdf_path: str = None):
    """주식 분석 실행"""

    print("=" * 60)
    print("🎯 버핏 스타일 주식 분석 시작")
    print("=" * 60)

    # RAG 초기화 (PDF 제공 시)
    if pdf_path:
        initialize_rag(pdf_path)

    # 워크플로우 생성
    app = create_investment_workflow()

    # 초기 상태
    initial_state = {
        "user_query": user_query,
        "market_data": {},
        "buffett_insights": [],
        "final_analysis": "",
        "error": ""
    }

    # 실행
    result = app.invoke(initial_state)

    print("\n" + "=" * 60)
    print("📊 분석 결과")
    print("=" * 60)
    print(result["final_analysis"])

    if result.get("error"):
        print(f"\n⚠️ 경고: {result['error']}")

    return result

In [17]:
pdf_path = 'stockking.pdf'

In [18]:
result = analyze_stock("괜찮은 배당주를 추천해줘.",pdf_path)

🎯 버핏 스타일 주식 분석 시작
📄 PDF 로딩 중: stockking.pdf
✓ RAG 초기화 완료: 180개 청크
🔍 Perplexity로 정보 수집 중: '괜찮은 배당주를 추천해줘.'
✓ Perplexity 정보 수집 완료 (출처: 주요 금융 사이트)
📚 Perplexity 결과 분석 후 관련 철학 검색 중...
  생성된 검색 주제: ['"dividend sustainability"', '"long-term growth potential"', '"competitive advantage"', '"management quality"', '"risk assessment"']
🤖 OpenAI로 종합 분석 중...
✓ 분석 완료 (2493 글자)

📊 분석 결과
### 1. 회사 및 티커 확인
- **알트리아 그룹 (NYSE:MO)**
- **버라이즌 (NYSE:VZ)**
- **AT&T (NYSE:T)**
- **킨더 모건 (NYSE:KMI)**
- **존슨앤존슨 (NYSE:JNJ)**
- **셰브론 (NYSE:CVX)**
- **펩시코 (NYSE:PEP)**

### 2. 비즈니스 이해도 (1-5점)
- **알트리아 그룹 (4점)**: 담배 및 관련 제품 제조업체로, 비즈니스 모델이 명확합니다. 그러나 담배 산업의 규제와 건강 문제는 리스크로 작용할 수 있습니다.
- **버라이즌 (4점)**: 통신 서비스 제공업체로, 안정적인 수익 모델을 가지고 있습니다. 버핏의 "능력 범위" 원칙에 부합합니다.
- **AT&T (4점)**: 통신 및 미디어 회사로, 안정적인 수익원이 있습니다. 비즈니스 모델이 명확합니다.
- **킨더 모건 (3점)**: 에너지 인프라 회사로, 파이프라인 운영을 통해 수익을 창출합니다. 에너지 시장의 변동성이 리스크입니다.
- **존슨앤존슨 (5점)**: 헬스케어 및 소비재 분야에서 강력한 포트폴리오를 보유하고 있습니다. 비즈니스 모델이 명확하며, 버핏의 원칙에 부합합니다.
- **셰브론 (4점)**: 에너지 부문에서 글로벌 리더로, 비즈니

In [19]:
# ============================================
# PDF 처리 과정 시각화 함수
# ============================================
def inspect_pdf_processing(pdf_path: str):
    """PDF가 어떻게 로드되고 벡터화되었는지 상세 확인"""

    print("=" * 60)
    print("📋 PDF 처리 과정 분석")
    print("=" * 60)

    # 1. PDF 로드
    print("\n[1단계] PDF 로딩...")
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    print(f"✓ 총 페이지 수: {len(documents)}페이지")
    print(f"\n첫 번째 페이지 미리보기 (처음 500자):")
    print("-" * 60)
    print(documents[0].page_content[:500])
    print("-" * 60)

    # 2. 텍스트 분할
    print("\n[2단계] 텍스트 분할...")
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )
    splits = text_splitter.split_documents(documents)

    print(f"✓ 생성된 청크 수: {len(splits)}개")
    print(f"\n청크 예시 (처음 3개):")
    for i, split in enumerate(splits[:3]):
        print(f"\n--- 청크 #{i+1} ---")
        print(f"페이지: {split.metadata.get('page', 'N/A')}")
        print(f"길이: {len(split.page_content)} 글자")
        print(f"내용 미리보기:")
        print(split.page_content[:300] + "...")

    # 3. 벡터화 및 검색 테스트
    print("\n[3단계] 벡터 스토어 생성 및 검색 테스트...")
    embeddings = OpenAIEmbeddings()
    vector_store_test = FAISS.from_documents(splits, embeddings)

    # 테스트 쿼리들
    test_queries = [
        "investment philosophy",
        "competitive advantage",
        "경제적 해자"
    ]

    print(f"\n검색 테스트 결과:")
    for query in test_queries:
        print(f"\n🔍 검색어: '{query}'")
        results = vector_store_test.similarity_search(query, k=2)
        for j, doc in enumerate(results):
            print(f"  [{j+1}] 유사도 높은 청크 (페이지 {doc.metadata.get('page', 'N/A')}):")
            print(f"      {doc.page_content[:200]}...")

    # 4. 통계 정보
    print("\n" + "=" * 60)
    print("📊 요약 통계")
    print("=" * 60)
    total_chars = sum(len(split.page_content) for split in splits)
    avg_chunk_size = total_chars / len(splits)

    print(f"전체 문자 수: {total_chars:,} 글자")
    print(f"평균 청크 크기: {avg_chunk_size:.0f} 글자")
    print(f"최소 청크 크기: {min(len(s.page_content) for s in splits)} 글자")
    print(f"최대 청크 크기: {max(len(s.page_content) for s in splits)} 글자")

    return {
        "documents": documents,
        "splits": splits,
        "vector_store": vector_store_test
    }

In [20]:
inspection_result = inspect_pdf_processing(pdf_path)

📋 PDF 처리 과정 분석

[1단계] PDF 로딩...
✓ 총 페이지 수: 48페이지

첫 번째 페이지 미리보기 (처음 500자):
------------------------------------------------------------
Berkshire – Past, Present and Future
In the Beginning
On May 6, 1964, Berkshire Hathaway, then run by a man named Seabury Stanton, sent a letter to its
shareholders offering to buy 225,000 shares of its stock for $11.375 per share. I had expected the letter; I was
surprised by the price.
Berkshire then had 1,583,680 shares outstanding. About 7% of these were owned by Buffett Partnership
Ltd. (“BPL”), an investing entity that I managed and in which I had virtually all of my net worth. Shortly bef
------------------------------------------------------------

[2단계] 텍스트 분할...
✓ 생성된 청크 수: 180개

청크 예시 (처음 3개):

--- 청크 #1 ---
페이지: 0
길이: 917 글자
내용 미리보기:
Berkshire – Past, Present and Future
In the Beginning
On May 6, 1964, Berkshire Hathaway, then run by a man named Seabury Stanton, sent a letter to its
shareholders offering to buy 225,000 shares of its stock f